In [1]:
import os
import pickle
from os.path import join
from copy import deepcopy
from collections import defaultdict, Counter
from datetime import datetime
from dateutil.relativedelta import relativedelta

import numpy as np
import pandas as pd
from tqdm import tqdm

import sklearn as sk
from scipy.optimize import linear_sum_assignment
from sklearn.metrics.pairwise import cosine_similarity

import torch
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, interact_manual

import bertopic
from sentence_transformers import SentenceTransformer

from proj_utils import *

/home/seungwoong.ha/anaconda3/envs/collmind/lib/python3.11/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/seungwoong.ha/anaconda3/envs/collmind/lib/python3.11/site-packages/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/seungwoong.ha/anaconda3/envs/collmind/lib/pyth

# 1. Topic frequency

## Preparation

In [2]:
# read csv file with pandas
collection_name = 'Gatewaypundit'
model_name = MODEL_NAMES[collection_name]
df = pd.read_csv(join('ctfidf', collection_name.lower(), 'topics_per_month.csv'))

In [3]:
# Sum all frequency for each 'month' and return a new dataframe
total_frequency = df.groupby('Month').sum()['Frequency']
# Normalize frequency of each topic by the total frequency of each month and name it as 'norm_freq'
df['norm_freq'] = df.apply(lambda row: row['Frequency'] / total_frequency[row['Month']], axis=1)
# ranking of each topics for each month and name it as 'rank'
df['rank'] = df.groupby('Month')['norm_freq'].rank(ascending=False)

topic_num = len(df['Topic'].unique())-1 # excluding -1

In [4]:
# remove topic -1 from df and name it as 'df2'
df2 = df[df['Topic'] != -1]

In [5]:
# Do the same thing as above
total_frequency2 = df2.groupby('Month').sum()['Frequency']
df2['norm_freq'] = df.apply(lambda row: row['Frequency'] / total_frequency2[row['Month']], axis=1)
df2['rank'] = df2.groupby('Month')['norm_freq'].rank(ascending=False)


## Visualization

### Total frequency

In [ ]:
total_frequency.plot(kind='bar', figsize=(15, 10), title='Total Frequency per Month')

In [ ]:
total_frequency2.plot(kind='bar', figsize=(15, 10), title='Total Frequency per Month')

### Number of unique topics per month

In [ ]:
# plot how many unique topics there are per month from df2
fig = plt.figure(figsize=(20, 10))
unique_topics = df2.groupby("Month").agg({"Topic": "nunique"})

# draw unique_topics on fig
ax1 = fig.add_subplot(111)
ax1.plot(unique_topics, label='unique topics', color='blue')
# turn x-label 90 degree
plt.xticks(rotation=90)

topic_diff = []
month_list = df2['Month'].unique()
for i in range(len(month_list)-1):
    topic_diff.append(len(set(df2[df2['Month']==month_list[i+1]]['Topic']) - set(df2[df2['Month']==month_list[i]]['Topic'])))
    
ax2 = plt.twinx()
ax2.plot(topic_diff, label='topic diff', color='red')
fig.legend(loc="upper right", bbox_to_anchor=(1,1), bbox_transform=ax1.transAxes)


### Ranking of each month

In [ ]:
# take month as 'month' and plot the frequency of each topic sorted by 'rank' for that month. show only top N topics.
def plot_month(df, month, N=20):
    df[df['Month'] == month].sort_values('rank').head(N).plot(kind='bar', x='Topic', y='norm_freq', figsize=(10, 7), fontsize=12, title='Frequency of topics for month {}'.format(month))
    
    plt.xlabel('Topic', fontsize=14)
    plt.ylabel('normalized frequency', fontsize=14)
    
    # Get the text of current x tick labels and makes it as a list
    labels = [item.get_text() for item in plt.gca().get_xticklabels()]

    # Adding keywords at the end of the x tick labels
    keywords = (df[df['Month'] == month]['Words'].str.split(',').str[0])[:N]   
    labels = [f'{labels[i]}, {keywords.iloc[i]}' for i in range(N)]
    plt.xticks(range(N), labels, rotation=45)                

In [ ]:
# with plot_month method, make a widget that shows the frequency of each topic sorted by 'rank' for each month.

months = df2['Month'].unique()
@interact
def plot_month_widget(month=months, N=20):
    plot_month(df2, month, N)

In [ ]:
# draw plot of topic versus frequency for all monght aggregated
freq = df2.groupby('Topic').sum()['Frequency'].plot(kind='bar', figsize=(10, 7), title='Frequency of topics for all months aggregated')
plt.gca().xaxis.set_major_locator(plt.MultipleLocator(10))
plt.xticks(rotation=90)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

# find the best fitting curve for this histogram
from scipy.optimize import curve_fit

def func(x, a, b, c):
    return a*np.exp(-b*x) + c

freq = df2.groupby('Topic').sum()['Frequency'].values
freq = freq[np.argsort(freq)[::-1]]
xdata = np.arange(len(freq))+1
popt, pcov = curve_fit(func, xdata, freq)
print(popt)
plt.bar(range(len(freq)), freq)
plt.plot(xdata, func(xdata, *popt), 'r-')
plt.title(f"Histogram of Comment Topic Numbers")
plt.xlabel("Topic Number")
plt.ylabel("Frequency")

formula = f'y = {popt[0]:.2f} * exp(-{popt[1]:.2f}x) + {popt[2]:.2f}'
plt.annotate(f'{formula}', xy=(len(freq)/2, max(freq)/2), xytext=(len(freq)/2, max(freq)/2), ha='center', va='center', fontsize=12, color='white', bbox=dict(facecolor='black', alpha=0.8))

plt.show()


### Ranking / frequency of each topic

In [ ]:
# take topic as 'topic' and plot the frequency of each topic for all months in the df.
def plot_topic(df, topic, ytype):
    months = df['Month'].unique()
    df[df['Topic'] == topic].plot(kind='bar', x='Month', y=ytype, figsize=(15, 10), fontsize=12, title='Frequency of topic {}'.format(topic))
    
    plt.xlabel('Month', fontsize=14)
    plt.ylabel(ytype, fontsize=14)

In [ ]:
# with plot_month method, make a widget that shows the frequency of each topic sorted by 'rank' for each month.

topics = df2['Topic'].unique()
@interact
def plot_topic_widget(df=df2, topic=topics, ytype=['norm_freq', 'rank']):
    plot_topic(df2, topic, ytype)

# 2. Topic embeddings

## Preparation

In [16]:
date_list = [DATE_RANGES[collection_name][0].strftime('%m%y')]
while True:
    next_date = next_month(date_list[-1])
    if next_date == DATE_RANGES[collection_name][1].strftime('%m%y'):
        break
    else:
        date_list.append(next_date)

In [17]:
total_mean_embedding_list= []
total_nonexist_index_list = []
for date in date_list:
    next_date = next_month(date)
    print(date)
    
    embedding_foler = '/data/comments/valentin/sentence-embeddings/'
    embedding_path = embedding_foler + f'{collection_name.lower()}/bert-emb-{date}-{next_date}.pt'
    embedding = torch.load(embedding_path)
    
    # check if embedding is on GPU and move it to CPU if it is
    
    
    topic_folder = 'transform'
    topic_path = join(topic_folder, collection_name.lower(), model_name, f'batch-{date}.arrow')
    topic = pd.read_feather(topic_path)
    
    if embedding['embeddings'].device.type == 'cuda':
        embedding['embeddings']= embedding['embeddings'].to('cpu')
    topic['embeddings'] = embedding['embeddings'].numpy().tolist()
    
    mean_embedding_list = []
    nonexist_index_list = []
    for i in range(topic_num):  # excluding topic -1
        current = topic[topic['topic']==i]
        if len(current)>0:
            averaged = np.mean(np.stack(current['embeddings'], axis=0), axis=0)
        else:
            nonexist_index_list.append(i)
            averaged = np.zeros(384)  # dim. of embeddings
        mean_embedding_list.append(averaged)

    print(nonexist_index_list)
    mean_embedding_list = np.array(mean_embedding_list)
    total_mean_embedding_list.append(mean_embedding_list)
    total_nonexist_index_list.append(nonexist_index_list)
    
total_mean_embedding_list = np.array(total_mean_embedding_list)
# save total_mean_embedding_list and total_nonexist_index_list as pickle
with open(join('transform', collection_name.lower(), model_name, 'total_mean_embedding_list.pickle'), 'wb') as f:
    pickle.dump(total_mean_embedding_list, f)
with open(join('transform', collection_name.lower(), model_name, 'total_nonexist_index_list.pickle'), 'wb') as f:
    pickle.dump(total_nonexist_index_list, f)

0115
[30, 40, 49, 69, 72, 81, 83, 87, 89, 90, 91, 92, 93, 116, 118, 125, 133, 135, 139, 140, 141, 142, 145, 148, 157, 163, 170, 173, 175, 178, 192, 193, 195, 199, 209, 211, 213, 216, 218, 222, 223, 224]
0215
[40, 45, 63, 69, 72, 81, 83, 88, 90, 91, 93, 106, 116, 118, 133, 139, 140, 141, 145, 148, 151, 163, 166, 170, 173, 177, 192, 193, 199, 200, 205, 209, 211, 213, 216, 218, 219, 222, 224, 229, 230]
0315
[30, 40, 63, 69, 81, 83, 86, 90, 93, 102, 116, 133, 142, 145, 148, 159, 164, 170, 173, 178, 180, 181, 192, 193, 200, 201, 205, 210, 214, 216, 218, 224]
0415
[30, 45, 49, 51, 63, 69, 74, 81, 87, 91, 92, 93, 106, 116, 124, 125, 127, 133, 139, 140, 141, 145, 159, 163, 164, 170, 173, 175, 177, 178, 186, 192, 200, 205, 210, 211, 213, 218, 224]
0515
[40, 45, 51, 69, 74, 76, 78, 81, 91, 103, 106, 116, 118, 125, 140, 141, 145, 148, 151, 157, 159, 164, 170, 175, 177, 181, 186, 195, 200, 205, 210, 216, 217, 218]
0615
[18, 30, 45, 46, 72, 76, 81, 83, 86, 90, 91, 93, 102, 106, 116, 125, 135, 139, 

In [ ]:
# load total_mean_embedding_list  and total_nonexist_index_list from pickle
with open(join('transform', collection_name.lower(), model_name, 'total_mean_embedding_list.pickle'), 'rb') as f:
    total_mean_embedding_list = pickle.load(f)
with open(join('transform', collection_name.lower(), model_name, 'total_nonexist_index_list.pickle'), 'rb') as f:
    total_nonexist_index_list = pickle.load(f)
    

In [ ]:
# for each element of total_mean_embedding_list, remove all-zero rows (non-existing topics) and concatenate all of them.
embedding_list_tsne = []

for i, mean_embedding in enumerate(total_mean_embedding_list):
    mean_embedding = np.delete(mean_embedding, total_nonexist_index_list[i], axis=0)
    embedding_list_tsne.append(mean_embedding)
    
embedding_list_tsne = np.concatenate(embedding_list_tsne, axis=0)

In [ ]:
X_embedded = sk.manifold.TSNE(n_components=2).fit_transform(embedding_list_tsne)

In [ ]:
total_tsne_list = []
previous_list = np.array([[0, 0]] * (topic_num))
counter = 0
for i in range(len(date_list)):
    print(i)
    tsne_list = []
    nonexist_index_list = deepcopy(total_nonexist_index_list[i])
    if len(nonexist_index_list)==0:
        tsne_list = X_embedded[counter:counter+topic_num]
        previous_list = X_embedded[counter:counter+topic_num]
        counter+=topic_num
    else:
        current_index = nonexist_index_list[0]
        for j in range(topic_num):
            if j==current_index:
                tsne_list.append(previous_list[j])
                nonexist_index_list.pop(0)
                if len(nonexist_index_list)>0:
                    current_index = nonexist_index_list[0]
            else:
                tsne_list.append(X_embedded[counter])
                previous_list[j] = X_embedded[counter]
                counter+=1
            
    assert len(tsne_list)==topic_num
    total_tsne_list.append(np.array(tsne_list))
    
total_tsne_list = np.array(total_tsne_list)

In [ ]:
# save total_tsne_list
np.save(f'tsne_{collection_name}.npy', total_tsne_list)

## Visualization

In [ ]:
total_tsne_list = np.load('tsne.npy')

In [ ]:
# visualizing total_tsne_list, which has 92 images of topic_num points with 2d coordinates, with widgets.
# add toggle button to show the movement of points
# set default i to 0

jet_colors = plt.get_cmap('jet')(np.linspace(0, 1, topic_num))

@interact
def show_tsne(i=(0, len(date_list)-1), movement=True, total=True, focal=(0, topic_num)):
    fig = plt.figure(figsize=(5, 4), dpi=200)
    ax = fig.add_subplot(111)
    
    # Assign every point a different color on the jet colormap
    if total: 
        im = ax.scatter(total_tsne_list[i][:, 0], total_tsne_list[i][:, 1], c=np.arange(topic_num), cmap='jet', s=5)
        # if i!=0, plot the previous points with alpha=0.2 to show the movement of points
        if movement and i!=0:
            ax.scatter(total_tsne_list[i-1][:, 0], total_tsne_list[i-1][:, 1], c=np.arange(topic_num), cmap='jet', s=5, alpha=0.2)
            # draw arrows to show the movement of points with alpha 0.2, with arrowhead length 0.1
            for j in range(topic_num):
                ax.arrow(total_tsne_list[i-1][j, 0], total_tsne_list[i-1][j, 1], total_tsne_list[i][j, 0]-total_tsne_list[i-1][j, 0], total_tsne_list[i][j, 1]-total_tsne_list[i-1][j, 1], color=jet_colors[j], head_width=1.5, alpha=0.2)
    else:
        # scatter a single point (focal) with color of focal point with jet colormap
    
        im = ax.scatter(total_tsne_list[i][:, 0], total_tsne_list[i][:, 1], c=np.arange(topic_num), cmap='jet', s=5, alpha=0.2)
        im2 = ax.scatter(total_tsne_list[i][focal, 0], total_tsne_list[i][focal, 1], c=jet_colors[focal], s=10, alpha=1)
        # if i!=0, plot the previous points with alpha=0.2 to show the movement of points
        if movement and i!=0:
            ax.scatter(total_tsne_list[i-1][focal, 0], total_tsne_list[i-1][focal, 1], c=jet_colors[focal], s=10, alpha=0.5)
            # draw arrows to show the movement of points with alpha 0.2, with arrowhead length 0.1
            ax.arrow(total_tsne_list[i-1][focal, 0], total_tsne_list[i-1][focal, 1], total_tsne_list[i][focal, 0]-total_tsne_list[i-1][focal, 0], total_tsne_list[i][focal, 1]-total_tsne_list[i-1][focal, 1], head_width=2.5, alpha=1)

    
    ax.set_xlim(-120, 120)
    ax.set_ylim(-120, 120)
    # Add a colorbar to show the meaning of the colors
    ax.set_xlabel('TSNE 1')
    ax.set_ylabel('TSNE 2')
    ax.set_title(f'TSNE of {topic_num} topics in {date_list[i] + "/" + next_month(date_list[i])}')
    
    # add a colorbar name
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Topic ID')
    
    plt.show()
# 174

# 3. Dist. and Corr. of Freq. and Sim.

In [ ]:
topic_model = (BERTopic.load(join('model', collection_name.lower(), model_name), embedding_model="all-MiniLM-L6-v2"))
embeddings = topic_model.topic_embeddings_

In [ ]:
# Distance

distance_matrix = np.zeros((topic_num, topic_num))
for i in range(topic_num):
    for j in range(topic_num):
        distance_matrix[i, j] = np.sqrt(np.sum((embeddings[i] - embeddings[j])**2))

distance_sum = np.sum(distance_matrix, axis=1)

In [ ]:
plt.plot(np.arange(len(distance_sum)-1), distance_sum[1:])

In [ ]:
# cos_sim
cos_sim_matrix = cosine_similarity(embeddings)
cos_sim_avg = np.mean(cos_sim_matrix, axis=1)

In [ ]:
plt.plot(np.arange(len(cos_sim_avg)-1), cos_sim_avg[1:])

In [ ]:
# get cos_sim matrix of each month
cos_sim_matrix_list = []
for i in range(len(date_list)):
    cos_sim_matrix_list.append(cosine_similarity(total_mean_embedding_list[i])) 
    
# get cos_sim_avg of each month
cos_sim_avg_list = []
for i in range(len(date_list)):
    cos_sim_avg_list.append(np.mean(cos_sim_matrix_list[i], axis=1))
    
# make widget to draw scatter plot of cos_sim_avg of each month, with topic frequency as x axis and cos_sim_avg as y axis
@interact
def plot_cos_sim_avg(i=(0, len(date_list)-1), remove_zero = False, mode=['rank', 'Frequency']):
    plt.figure(figsize=(10, 7))
    # convert date %m%y into yyyy-mm-dd
    month = datetime.strptime(date_list[i], '%m%y').strftime('%Y-%m-%d')
    if remove_zero:
        plt.scatter(df[df['Month'] == month][mode].values[2:], cos_sim_avg_list[i][df[df['Month'] == month]['Topic'].values][2:], s=5)
    else:
        plt.scatter(df[df['Month'] == month][mode].values[1:], cos_sim_avg_list[i][df[df['Month'] == month]['Topic'].values][1:], s=5)
    plt.xlabel(mode)
    plt.ylabel('Cosine Similarity')
    plt.title(f'Cosine Similarity vs Topic Frequency in {date_list[i]}')
    plt.show()

## 4. Edge weight distribution

In [ ]:
# calculate cosine similarity of two vectors
def cos_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
topic_folder = 'ctfidf'
topic_path = join(topic_folder, collection_name.lower(), f'ctfidf-{date}.arrow')
topic = pd.read_feather(topic_path)
from scipy.sparse import coo_matrix

coo = coo_matrix((topic['x'], (topic['row'], topic['col'])), dtype=np.float32)

In [ ]:
mean_weight_list = []
distance_matrix_list = []
for i in range(len(total_mean_embedding_list)):
    x = total_mean_embedding_list[i]

    # for x, I want to get a distance matrix between all pairs

    def get_distance_matrix(x):
        distance_matrix = np.zeros((len(x), len(x)))
        for i in range(len(x)):
            for j in range(len(x)):
                #distance_matrix[i, j]= np.sqrt(np.sum((x[i] - x[j])**2))
                if (np.linalg.norm(x[i]) * np.linalg.norm(x[j])) != 0:
                    distance_matrix[i, j] = cos_sim(x[i], x[j])
        return distance_matrix

    distance_matrix = get_distance_matrix(x)
    distance_matrix = np.delete(distance_matrix, total_nonexist_index_list[i], axis=0)
    distance_matrix = np.delete(distance_matrix, total_nonexist_index_list[i], axis=1)
    distance_matrix_list.append(distance_matrix)
    mean_weight = np.sum(distance_matrix, axis=0) / (len(distance_matrix)-1)
    mean_weight_list.append(mean_weight)

In [ ]:
# make a widget that draws distribution of distance_matrix of each month
@interact
def plot_distance_matrix(i=(0, len(date_list)-1)):
    plt.figure(figsize=(10, 7))
    # flatten the distance matrix and exclude the diagonal term
    flattened = distance_matrix_list[i].flatten()
    flattened = flattened[flattened != 1]
    plt.hist(flattened, bins=20)
    plt.xlabel('Cosine Similarity')
    plt.ylabel('Frequency')
    plt.title(f'Distribution of Cosine Similarity in {date_list[i]}')
    
    plt.show()

In [ ]:
# make a widget that draws histogram of mean_weight_list[i] for each i
@interact
def plot_mean_weight(i=(0, len(date_list)-1)):
    plt.figure(figsize=(10, 7))
    x = distance_matrix_list[i]
    plt.hist(np.ravel(x[1:, 1:])[~np.eye(x.shape[0]-1, dtype=bool).flatten()], bins=20)
    plt.xlabel('Embedding distance')
    plt.ylabel('Frequency')
    plt.title(f'Mean Weight of {topic_num} topics in {date_list[i]}')
    plt.show()

In [ ]:
# draw a plot of variance of mean_weight_list
plt.figure(figsize=(10, 7))
plt.plot(np.arange(len(distance_matrix_list)), [np.mean(np.ravel(x[1:, 1:])[~np.eye(x.shape[0]-1, dtype=bool).flatten()]) for x in distance_matrix_list])

In [ ]:
# draw a plot of variance of mean_weight_list
plt.figure(figsize=(10, 7))
plt.plot(np.arange(len(distance_matrix_list)), [np.var(np.ravel(x[1:, 1:])[~np.eye(x.shape[0]-1, dtype=bool).flatten()]) for x in distance_matrix_list])

In [ ]:
# make a widget that draws histogram of mean_weight_list[i] for each i
@interact
def plot_mean_weight(i=(0, len(date_list)-1)):
    plt.figure(figsize=(10, 7))
    plt.hist(mean_weight_list[i], bins=20)
    plt.xlabel('Mean Weight')
    plt.ylabel('Frequency')
    plt.title(f'Mean Weight of {topic_num} topics in {date_list[i]}')
    plt.show()

## 5. Corr. between freq. and similarity of topics


In [ ]:
# get the frequency of each topic at each month
df_freq = df.groupby([df['Month'].apply(lambda x: x[:-2] + '01'), 'Topic'])['Frequency'].sum().reset_index()
# change df_freq 'Month' form (YYYY-MM-DD) to (MMYY)
df_freq['Month'] = pd.to_datetime(df_freq['Month']).apply(lambda x: x.strftime('%m%y'))

In [ ]:
# get the frequency of each topic at each month
df_freq = df.groupby([df['Month'].apply(lambda x: x[:-2] + '01'), 'Topic'])['Frequency'].sum().reset_index()
# change df_freq 'Month' form (YYYY-MM-DD) to (MMYY)
df_freq['Month'] = pd.to_datetime(df_freq['Month']).apply(lambda x: x.strftime('%m%y'))

order_index_list = []
# sort the distance matrix by the frequency of each topic at each month
distance_matrix_sorted_list = []
for i in range(len(distance_matrix_list)):
    freq = df_freq[df_freq['Month'] == date_list[i]]['Frequency'].values[1:]  # exclude topic -1
    order_index = np.argsort(freq)[::-1]
    distance_matrix_sorted = distance_matrix_list[i][order_index]
    distance_matrix_sorted = distance_matrix_sorted[:, order_index]
    order_index_list.append(order_index)
    distance_matrix_sorted_list.append(distance_matrix_sorted)

@interact
def plot_distance_matrix(i=(0, len(date_list)-1)):
    plt.imshow(distance_matrix_sorted_list[i], cmap='hot')
    # colorbar
    plt.colorbar()
    plt.title(f"Distance Matrix for {date_list[i]}")
    plt.show()
    

## 6. Comment Distribution

In [ ]:
from collections import Counter

collection_name = "Atlantic"

start_date, end_date = DATE_RANGES[collection_name]
time_period = relativedelta(end_date, start_date)
messages, topics, timestamps = [], [], []
art_id_counter = Counter()
art_id_dict = defaultdict(list)

for embeddings_month, current_date in tqdm(gen_sent_embeddings(collection_name, start_date, end_date), total=12 * time_period.years + time_period.months):
    comments_df = query_comments_by_id("Comments", collection_name, embeddings_month["_id"],  select_columns=["_id", "raw_message", "createdAt", "art_id"]) \
        .set_index("_id").loc[embeddings_month["_id"]]
    temp = Counter(comments_df['art_id'].values)
    
    # make a dictionary from comments_df, where the key is art_id and value is the lis of _id shares the same art_id
    for i, row in comments_df.iterrows():
        art_id_dict[row['art_id']].append(i)
        
    art_id_counter += temp

In [ ]:
threshold = 0
art_id_counter_filtered = {k: v for k, v in art_id_counter.items() if v >= threshold}
    
freq, bins = np.histogram(list(art_id_counter_filtered.values()), bins=np.arange(max(art_id_counter_filtered.values())+1))

In [ ]:
plt.hist(list(art_id_counter_filtered.values()), bins=np.arange(max(art_id_counter_filtered.values())+1), density=True)